# Sistema de Recuperación de Información - SciFact

**Corpus asignado:** SciFact (BeIR)  
**Objetivo:** Indexar documentos científicos, recuperar resultados con TF-IDF + similitud coseno y BM25, evaluar con P@10, R@10 y NDCG@10 sobre los qrels de test, y exponer la búsqueda como servicio web.  

Este notebook implementa la parte de indexación, recuperación y evaluación **sin usar embeddings**. Toda la búsqueda es por términos.

## 1. Instalación de dependencias y recursos NLTK

Instala las dependencias en el entorno del kernel y ejecuta el notebook desde la carpeta que contiene `examen1bim.ipynb`.

```bash
python -m pip install datasets scikit-learn nltk numpy scipy pandas fastapi uvicorn
```

La preparación usa SciFact y los recursos de NLTK. Las celdas de la sección 9 guardan el índice listo para cargar y los recursos del tokenizador en `despliegue/`.

In [1]:
import gzip
import json
import math
import re
from pathlib import Path

import nltk
import numpy as np
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from collections import defaultdict

import pandas as pd
from datasets import load_dataset
from nltk.corpus import stopwords

for recurso, ubicacion in (
    ('punkt_tab', 'tokenizers/punkt_tab/english/'),
    ('stopwords', 'corpora/stopwords'),
):
    try:
        nltk.data.find(ubicacion)
    except LookupError:
        nltk.download(recurso, quiet=True, raise_on_error=True)

print('Dependencias cargadas correctamente')

Dependencias cargadas correctamente


## 2. Carga de SciFact y verificación de IDs

Se cargan:  
- **Corpus:** documentos con `_id`, `title`, `text`.  
- **Queries:** consultas con `_id`, `text`.  
- **Qrels:** juicios de relevancia de test con `query-id`, `corpus-id`, `score`.  

Se verifica que cada `query-id` y `corpus-id` de los qrels exista en los conjuntos correspondientes.

In [3]:
# Cargar datasets
corpus = load_dataset('BeIR/scifact', 'corpus', split='corpus')
queries = load_dataset('BeIR/scifact', 'queries', split='queries')
qrels = load_dataset('BeIR/scifact-qrels', split='test')

print('Corpus:', corpus)
print('Queries:', queries)
print('Qrels test:', qrels)
print()
print('Ejemplo corpus:', corpus[0])
print('Ejemplo query:', queries[0])
print('Ejemplo qrel:', qrels[0])

Corpus: Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 5183
})
Queries: Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 1109
})
Qrels test: Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 339
})

Ejemplo corpus: {'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, earl

In [4]:
# Verificar correspondencia de IDs
corpus_ids = set(doc['_id'] for doc in corpus)
query_ids = set(q['_id'] for q in queries)
qrel_corpus_ids = set(str(q['corpus-id']) for q in qrels)
qrel_query_ids = set(str(q['query-id']) for q in qrels)

print(f'Total documentos en corpus: {len(corpus_ids)}')
print(f'Total consultas: {len(query_ids)}')
print(f'Total qrels de test: {len(qrels)}')
print(f'Queries distintas en qrels: {len(qrel_query_ids)}')
print(f'Documentos de qrels ausentes en corpus: {len(qrel_corpus_ids - corpus_ids)}')
print(f'Queries de qrels ausentes en queries: {len(qrel_query_ids - query_ids)}')
assert qrel_corpus_ids.issubset(corpus_ids), 'Hay corpus-id en qrels que no existen en el corpus'
assert qrel_query_ids.issubset(query_ids), 'Hay query-id en qrels que no existen en queries'
print('\n Todos los IDs de los qrels corresponden a documentos y consultas válidas.')

Total documentos en corpus: 5183
Total consultas: 1109
Total qrels de test: 339
Queries distintas en qrels: 300
Documentos de qrels ausentes en corpus: 0
Queries de qrels ausentes en queries: 0

 Todos los IDs de los qrels corresponden a documentos y consultas válidas.


## 3. Preprocesamiento e índice invertido

**Decisiones de preprocesamiento:**  
1. **Normalización:** conversión a minúsculas y eliminación de caracteres no alfanuméricos.  
2. **Tokenización:** `word_tokenize` de NLTK.  
3. **Stopwords:** lista en inglés de NLTK.  
4. **Stemming:** algoritmo de Porter.  
5. **Filtrado:** se conservan solo tokens alfabéticos de más de 2 caracteres.  

Cada documento se indexa como concatenación de `title + ' ' + text`, conservando su `_id` original.

In [5]:
class Buscador:
    """Recupera documentos completos con TF-IDF o BM25."""

    def __init__(self, doc_ids, doc_titles, doc_texts, stop_words):
        if len(doc_ids) != len(set(doc_ids)):
            raise ValueError('Los IDs del corpus deben ser únicos')
        self.doc_ids = doc_ids
        self.doc_titles = doc_titles
        self.doc_texts = doc_texts
        self.stop_words = set(stop_words)
        self.stemmer = PorterStemmer()

    def preprocess(self, text):
        """Normaliza, tokeniza, elimina stopwords y aplica stemming."""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return [
            self.stemmer.stem(tok)
            for tok in word_tokenize(text)
            if tok.isalpha() and tok not in self.stop_words and len(tok) > 2
        ]

    def ajustar_tfidf(self, tokenized_docs):
        preprocessed_docs = [' '.join(tokens) for tokens in tokenized_docs]
        self.tfidf_vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(preprocessed_docs)

    def search_tfidf(self, query, top_k=10):
        """Devuelve pares de ID y similitud coseno con puntaje positivo."""
        q_text = ' '.join(self.preprocess(query))
        q_vec = self.tfidf_vectorizer.transform([q_text])
        similarities = cosine_similarity(q_vec, self.tfidf_matrix).flatten()
        top_idx = np.argsort(similarities)[::-1][:top_k]
        results = []
        for idx in top_idx:
            score = float(similarities[idx])
            if score <= 0:
                break
            results.append((self.doc_ids[idx], score))
        return results

    def ajustar_bm25(self, tokenized_docs, inverted_index):
        self.tokenized_docs = tokenized_docs
        self.inverted_index = inverted_index
        self.doc_lengths = np.array([len(tokens) for tokens in tokenized_docs], dtype=np.float64)
        self.avgdl = float(np.mean(self.doc_lengths))
        N = len(tokenized_docs)
        df = {term: len(docs) for term, docs in inverted_index.items()}
        self.idf = {
            term: math.log((N - n + 0.5) / (n + 0.5) + 1e-10)
            for term, n in df.items()
        }
        self.k1 = 1.5
        self.b = 0.75

    def search_bm25(self, query, top_k=10):
        """Devuelve pares de ID y puntaje BM25 positivo."""
        q_tokens = self.preprocess(query)
        scores = np.zeros(len(self.doc_ids), dtype=np.float64)
        for term in q_tokens:
            if term not in self.inverted_index:
                continue
            idf_term = self.idf[term]
            for doc_idx in self.inverted_index[term]:
                f = self.tokenized_docs[doc_idx].count(term)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * self.doc_lengths[doc_idx] / self.avgdl)
                scores[doc_idx] += idf_term * f * (self.k1 + 1) / denom
        top_idx = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_idx:
            score = float(scores[idx])
            if score <= 0:
                break
            results.append((self.doc_ids[idx], score))
        return results

In [6]:
# Preparar documentos conservando IDs originales y orden del corpus
doc_ids = [doc['_id'] for doc in corpus]
doc_titles = {doc['_id']: doc['title'] or '' for doc in corpus}
doc_texts = {doc['_id']: doc['text'] or '' for doc in corpus}
buscador = Buscador(doc_ids, doc_titles, doc_texts, stopwords.words('english'))
tokenized_docs = [
    buscador.preprocess((doc_titles[did] + ' ' + doc_texts[did]).strip())
    for did in doc_ids
]

print(f'Documentos indexados: {len(doc_ids)}')
print(f'Ejemplo de tokens (doc {doc_ids[0]}): {tokenized_docs[0][:30]}')

Documentos indexados: 5183
Ejemplo de tokens (doc 4983): ['microstructur', 'develop', 'human', 'newborn', 'cerebr', 'white', 'matter', 'assess', 'vivo', 'diffus', 'tensor', 'magnet', 'reson', 'imag', 'alter', 'architectur', 'cerebr', 'white', 'matter', 'develop', 'human', 'brain', 'affect', 'cortic', 'develop', 'result', 'function', 'disabl', 'line', 'scan']


In [7]:
# Construir índice invertido: término -> lista de índices de documentos
inverted_index = defaultdict(list)
for idx, tokens in enumerate(tokenized_docs):
    for term in set(tokens):
        inverted_index[term].append(idx)

vocab_size = len(inverted_index)
print(f'Tamaño del vocabulario: {vocab_size}')
print(f'Ejemplo de índice invertido: {list(inverted_index.keys())[0]} -> {inverted_index[list(inverted_index.keys())[0]][:5]}')

Tamaño del vocabulario: 19856
Ejemplo de índice invertido: preterm -> [0, 66, 1724, 1909, 1919]


## 4. Modelo TF-IDF con similitud coseno

Se entrena un `TfidfVectorizer` de scikit-learn sobre los documentos preprocesados ya unidos como cadenas. Para consultar, se preprocesa la consulta de la misma forma y se calcula la similitud coseno con todos los documentos.

In [8]:
buscador.ajustar_tfidf(tokenized_docs)
search_tfidf = buscador.search_tfidf
print(f'Matriz TF-IDF: {buscador.tfidf_matrix.shape}')

sample_query = next(q for q in queries if q['_id'] == '1')['text']
print('Consulta de prueba:', sample_query)
print('Top 3 TF-IDF:', search_tfidf(sample_query, 3))

Matriz TF-IDF: (5183, 19855)
Consulta de prueba: 0-dimensional biomaterials show inductive properties.
Top 3 TF-IDF: [('10906636', 0.10632252789889068), ('10608397', 0.10197873840324424), ('31543713', 0.09677198724941677)]


## 5. Modelo BM25

Implementación manual sobre el mismo índice invertido. Parámetros estándar: `k1=1.5`, `b=0.75`. La fórmula usada es el score BM25 estándar:

```
score(q, d) = sum_t in q [ idf(t) * (f(t,d) * (k1 + 1)) / (f(t,d) + k1 * (1 - b + b * |d| / avgdl)) ]
idf(t) = log((N - n_t + 0.5) / (n_t + 0.5) + 1e-10)
```

In [9]:
buscador.ajustar_bm25(tokenized_docs, inverted_index)
search_bm25 = buscador.search_bm25
print('Top 3 BM25:', search_bm25(sample_query, 3))

Top 3 BM25: [('21257564', 9.749843781504081), ('18953920', 9.534339229867992), ('40212412', 9.224695024001619)]


## 6. Funciones de evaluación

Se implementan Precision@k, Recall@k y NDCG@k usando los qrels de test. NDCG se calcula manualmente con la fórmula estándar usando ganancias binarias.

In [10]:
# Convertir qrels a diccionario: query-id -> set de corpus-id relevantes
qrels_dict = defaultdict(set)
for qrel in qrels:
    qid = str(qrel['query-id'])
    cid = str(qrel['corpus-id'])
    if int(qrel['score']) > 0:
        qrels_dict[qid].add(cid)

def precision_at_k(retrieved, relevant, k=10):
    retrieved_k = retrieved[:k]
    if not retrieved_k:
        return 0.0
    return len(set(retrieved_k) & relevant) / k

def recall_at_k(retrieved, relevant, k=10):
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def dcg(relevances):
    return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))

def ndcg_at_k(retrieved, relevant, k=10):
    rel = [1 if doc_id in relevant else 0 for doc_id in retrieved[:k]]
    ideal = sorted([1] * len(relevant) + [0] * (k - len(relevant)), reverse=True)[:k]
    ideal_dcg = dcg(ideal)
    if ideal_dcg == 0:
        return 0.0
    return dcg(rel) / ideal_dcg

assert math.isclose(precision_at_k(['a', 'b', 'c', 'd'], {'a'}, k=10), 0.10)
assert precision_at_k([], {'a'}, k=10) == 0.0
print('P@10 con 4 recuperados y 1 relevante:', precision_at_k(['a', 'b', 'c', 'd'], {'a'}, k=10))
print('P@10 con lista vacía:', precision_at_k([], {'a'}, k=10))

print(f'Queries de test con qrels: {len(qrels_dict)}')
print(f'Ejemplo qrels (query 1): {qrels_dict["1"]}')

P@10 con 4 recuperados y 1 relevante: 0.1
P@10 con lista vacía: 0.0
Queries de test con qrels: 300
Ejemplo qrels (query 1): {'31715818'}


## 7. Evaluación de TF-IDF y BM25 sobre test

Se evalúan ambos modelos con las 300 consultas de test que tienen qrels. Se reportan Precision@10, Recall@10 y NDCG@10 promedios.

In [11]:
# Diccionario de consultas por id
queries_dict = {q['_id']: q['text'] for q in queries}

# Listas para métricas
tfidf_p10, tfidf_r10, tfidf_ndcg10 = [], [], []
bm25_p10, bm25_r10, bm25_ndcg10 = [], [], []

for qid in sorted(qrels_dict.keys(), key=int):
    query_text = queries_dict[qid]
    relevant = qrels_dict[qid]

    tfidf_results = search_tfidf(query_text, top_k=10)
    tfidf_retrieved = [did for did, _ in tfidf_results]
    tfidf_p10.append(precision_at_k(tfidf_retrieved, relevant, k=10))
    tfidf_r10.append(recall_at_k(tfidf_retrieved, relevant, k=10))
    tfidf_ndcg10.append(ndcg_at_k(tfidf_retrieved, relevant, k=10))

    bm25_results = search_bm25(query_text, top_k=10)
    bm25_retrieved = [did for did, _ in bm25_results]
    bm25_p10.append(precision_at_k(bm25_retrieved, relevant, k=10))
    bm25_r10.append(recall_at_k(bm25_retrieved, relevant, k=10))
    bm25_ndcg10.append(ndcg_at_k(bm25_retrieved, relevant, k=10))

# Promedios
metrics = {
    'TF-IDF': {
        'Precision@10': np.mean(tfidf_p10),
        'Recall@10': np.mean(tfidf_r10),
        'NDCG@10': np.mean(tfidf_ndcg10),
    },
    'BM25': {
        'Precision@10': np.mean(bm25_p10),
        'Recall@10': np.mean(bm25_r10),
        'NDCG@10': np.mean(bm25_ndcg10),
    }
}

df_metrics = pd.DataFrame(metrics).T
print(df_metrics)

# Guardar resultados numéricos para referencia
print('\nResumen numérico:')
print(json.dumps({k: {m: round(v, 4) for m, v in vals.items()} for k, vals in metrics.items()}, indent=2))

        Precision@10  Recall@10   NDCG@10
TF-IDF      0.075000   0.689222  0.533814
BM25        0.084333   0.767000  0.632472

Resumen numérico:
{
  "TF-IDF": {
    "Precision@10": 0.075,
    "Recall@10": 0.6892,
    "NDCG@10": 0.5338
  },
  "BM25": {
    "Precision@10": 0.0843,
    "Recall@10": 0.767,
    "NDCG@10": 0.6325
  }
}


## 8. Comparación e interpretación de resultados

Resultados obtenidos en esta ejecución sobre las 300 consultas de test (promedios):

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|
| TF-IDF + coseno | 0.0750 | 0.6892 | 0.5338 |
| BM25 | 0.0843 | 0.7670 | 0.6325 |
| Diferencia (BM25 − TF-IDF) | +0.0093 | +0.0778 | +0.0987 |

**Por qué Precision@10 es baja y Recall@10 alta.** Los qrels de test de SciFact son binarios y muy dispersos: hay 339 juicios relevantes para 300 consultas, de modo que la mayoría de las consultas tiene **un único documento relevante**. Precision@10 divide los relevantes recuperados entre k = 10, aunque el modelo devuelva menos de diez resultados. Por eso, cuando una consulta tiene un único documento relevante y este aparece entre los diez primeros, **Precision@10 vale 1/10 = 0.10** y **Recall@10 vale 1/1 = 1.0**; si no aparece, ambas valen 0. Con esta estructura de qrels, Precision@10 apenas puede superar 0.10 en promedio, y el cociente Recall@10 / Precision@10 (≈ 9.2 en TF-IDF y ≈ 9.1 en BM25) es coherente con algo más de un documento relevante por consulta.

**Conclusión de esta ejecución:**
- **BM25 supera a TF-IDF en las tres métricas.** Recupera el documento relevante entre los diez primeros en más consultas (Recall@10 0.767 frente a 0.689, es decir, unas 7.8 puntos porcentuales más), lo que se traduce en una Precision@10 mayor (0.0843 frente a 0.0750).
- **La mayor ventaja está en NDCG@10** (0.6325 frente a 0.5338, +0.099, ≈ 18 % relativo), mayor que la mejora relativa de Recall@10 (≈ 11 %). Esto indica que BM25 no solo encuentra más documentos relevantes, sino que además los coloca en posiciones más altas del ranking.
- Las diferencias de Precision@10 parecen pequeñas en valor absoluto solo porque su techo práctico es ≈ 0.10; en relación con ese techo, la mejora de BM25 es del mismo orden que la de Recall@10.
- En este corpus, la saturación de la frecuencia de términos (k1) y la normalización por longitud del documento (b) de BM25 resultan más eficaces que la ponderación TF-IDF con similitud coseno para ordenar resúmenes científicos.

## 9. Servicio web local con FastAPI

El notebook exporta la clase `Buscador` que se utilizó en la evaluación. El archivo comprimido guarda los documentos completos en el orden original del corpus, la matriz y los pesos de TF-IDF, y los datos del índice invertido de BM25. También guarda las stopwords y la carpeta `data/nltk_data/` incluye el tokenizador inglés de NLTK.

`GET /health` confirma que el buscador tiene el índice cargado y devuelve `{"status": "ok", "index_loaded": true, "documents": 5183}`; si el índice no estuviera cargado respondería HTTP 503. `POST /search` acepta `query`, `model` (`tfidf` o `bm25`) y `top_k` (entero mayor o igual que cero). El `snippet` contiene hasta 240 caracteres del texto original y se genera después de ordenar los documentos.

In [12]:
def guardar_indice(buscador, archivo):
    matriz = buscador.tfidf_matrix
    indice = {
        'doc_ids': buscador.doc_ids,
        'doc_titles': buscador.doc_titles,
        'doc_texts': buscador.doc_texts,
        'stop_words': sorted(buscador.stop_words),
        'tfidf': {
            'vocabulary': {term: int(col) for term, col in buscador.tfidf_vectorizer.vocabulary_.items()},
            'idf': buscador.tfidf_vectorizer.idf_.tolist(),
            'shape': matriz.shape,
            'data': matriz.data.tolist(),
            'indices': matriz.indices.tolist(),
            'indptr': matriz.indptr.tolist(),
        },
        'bm25': {
            'tokenized_docs': buscador.tokenized_docs,
            'inverted_index': buscador.inverted_index,
            'doc_lengths': buscador.doc_lengths.tolist(),
            'avgdl': buscador.avgdl,
            'idf': buscador.idf,
            'k1': buscador.k1,
            'b': buscador.b,
        },
    }
    with gzip.open(archivo, 'wt', encoding='utf-8', compresslevel=6) as salida:
        json.dump(indice, salida, ensure_ascii=False, separators=(',', ':'))


def cargar_indice(archivo):
    """Restaura los modelos y el corpus a partir del índice ya calculado."""
    recursos_nltk = str(archivo.parent / 'nltk_data')
    if recursos_nltk not in nltk.data.path:
        nltk.data.path.insert(0, recursos_nltk)
    nltk.data.find('tokenizers/punkt_tab/english/', paths=[recursos_nltk])

    with gzip.open(archivo, 'rt', encoding='utf-8') as entrada:
        indice = json.load(entrada)
    buscador = Buscador(indice['doc_ids'], indice['doc_titles'], indice['doc_texts'], indice['stop_words'])

    tfidf = indice['tfidf']
    buscador.tfidf_vectorizer = TfidfVectorizer(vocabulary=tfidf['vocabulary'])
    buscador.tfidf_vectorizer.idf_ = np.asarray(tfidf['idf'], dtype=np.float64)
    buscador.tfidf_matrix = csr_matrix(
        (tfidf['data'], tfidf['indices'], tfidf['indptr']),
        shape=tuple(tfidf['shape']),
        dtype=np.float64,
    )

    bm25 = indice['bm25']
    buscador.tokenized_docs = bm25['tokenized_docs']
    buscador.inverted_index = bm25['inverted_index']
    buscador.doc_lengths = np.asarray(bm25['doc_lengths'], dtype=np.float64)
    buscador.avgdl = bm25['avgdl']
    buscador.idf = bm25['idf']
    buscador.k1 = bm25['k1']
    buscador.b = bm25['b']
    return buscador

In [13]:
from importlib.metadata import version

despliegue = Path('despliegue').resolve()
directorio_datos = despliegue / 'data'
directorio_datos.mkdir(parents=True, exist_ok=True)

# Exportar las mismas definiciones ejecutadas en este notebook.
notebook = json.loads(Path('examen1bim.ipynb').read_text(encoding='utf-8'))
fuentes = [
    ''.join(celda['source']).rstrip()
    for celda in notebook['cells']
    if 'exportar-buscador' in celda.get('metadata', {}).get('tags', [])
]
modulo_buscador = '\n\n\n'.join(fuentes) + '\n'
compile(modulo_buscador, 'buscador.py', 'exec')
(despliegue / 'buscador.py').write_text(modulo_buscador, encoding='utf-8')

main_py = '''
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal

from fastapi import FastAPI, Response
from pydantic import BaseModel, Field

from buscador import cargar_indice


Modelo = Literal['tfidf', 'bm25']
ARCHIVO_INDICE = Path(__file__).resolve().parent / 'data' / 'indice_scifact.json.gz'


class Consulta(BaseModel):
    query: str
    model: Modelo
    top_k: int = Field(ge=0, strict=True)


class Resultado(BaseModel):
    rank: int
    document_id: str
    score: float
    title: str
    snippet: str


class RespuestaBusqueda(BaseModel):
    model: Modelo
    results: list[Resultado]


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.buscador = cargar_indice(ARCHIVO_INDICE)
    yield


app = FastAPI(title='Buscador SciFact', lifespan=lifespan)


@app.get('/health')
def health(response: Response):
    buscador = getattr(app.state, 'buscador', None)
    if buscador is None:
        response.status_code = 503
        return {'status': 'unavailable', 'index_loaded': False, 'documents': 0}
    return {'status': 'ok', 'index_loaded': True, 'documents': len(buscador.doc_ids)}


@app.post('/search', response_model=RespuestaBusqueda)
def search(consulta: Consulta):
    buscador = app.state.buscador
    ranking = buscador.search_tfidf if consulta.model == 'tfidf' else buscador.search_bm25
    results = []
    for rank, (doc_id, score) in enumerate(ranking(consulta.query, consulta.top_k), start=1):
        texto = buscador.doc_texts[doc_id]
        snippet = texto[:239].rstrip() + '…' if len(texto) > 240 else texto
        results.append(Resultado(
            rank=rank,
            document_id=doc_id,
            score=score,
            title=buscador.doc_titles[doc_id],
            snippet=snippet,
        ))
    return RespuestaBusqueda(model=consulta.model, results=results)
'''.lstrip()
compile(main_py, 'main.py', 'exec')
(despliegue / 'main.py').write_text(main_py, encoding='utf-8')

paquetes = ('fastapi', 'uvicorn', 'pydantic', 'nltk', 'numpy', 'scipy', 'scikit-learn')
requisitos = ''.join(f'{paquete}=={version(paquete)}\n' for paquete in paquetes)
(despliegue / 'requirements.txt').write_text(requisitos, encoding='utf-8')

punkt_ingles = nltk.data.find('tokenizers/punkt_tab/english/')
destino_punkt = directorio_datos / 'nltk_data' / 'tokenizers' / 'punkt_tab' / 'english'
destino_punkt.mkdir(parents=True, exist_ok=True)
for nombre in ('abbrev_types.txt', 'collocations.tab', 'ortho_context.tab', 'sent_starters.txt'):
    with punkt_ingles.join(nombre).open() as entrada:
        (destino_punkt / nombre).write_bytes(entrada.read())

archivo_indice = directorio_datos / 'indice_scifact.json.gz'
guardar_indice(buscador, archivo_indice)
print(f'Índice guardado con {len(buscador.doc_ids)} documentos')
for archivo in sorted(despliegue.rglob('*')):
    if archivo.is_file() and '__pycache__' not in archivo.parts:
        print(f'{archivo.relative_to(despliegue.parent).as_posix()} ({archivo.stat().st_size:,} bytes)')

Índice guardado con 5183 documentos
despliegue/.gitignore (9 bytes)
despliegue/.python-version (5 bytes)
despliegue/.vercel/project.json (116 bytes)
despliegue/.vercel/README.txt (520 bytes)
despliegue/.vercelignore (35 bytes)
despliegue/buscador.py (6,043 bytes)
despliegue/data/indice_scifact.json.gz (10,200,708 bytes)
despliegue/data/nltk_data/tokenizers/punkt_tab/english/abbrev_types.txt (619 bytes)
despliegue/data/nltk_data/tokenizers/punkt_tab/english/collocations.tab (594 bytes)
despliegue/data/nltk_data/tokenizers/punkt_tab/english/ortho_context.tab (236,303 bytes)
despliegue/data/nltk_data/tokenizers/punkt_tab/english/sent_starters.txt (241 bytes)
despliegue/main.py (1,929 bytes)
despliegue/requirements.txt (117 bytes)


### Verificación HTTP

La celda inicia Uvicorn en un proceso aparte desde `despliegue/` y lo detiene al terminar. Compara los rankings de la consulta 1 con los IDs y puntajes que se guardaron antes de implementar la API, con tolerancia absoluta de `1e-12`. También comprueba una consulta ajena a las consultas del corpus, otra sin coincidencias y la validación de los parámetros.

In [14]:
import os
import socket
import subprocess
import sys
import tempfile
import time
from urllib.error import HTTPError, URLError
from urllib.request import ProxyHandler, Request, build_opener

consulta_original = '0-dimensional biomaterials show inductive properties.'
rankings_originales = {'tfidf': [('10906636', 0.10632252789889068), ('10608397', 0.10197873840324424), ('31543713', 0.09677198724941677)], 'bm25': [('21257564', 9.749843781504081), ('18953920', 9.534339229867992), ('40212412', 9.224695024001619)]}
consulta_nueva = 'diffusion imaging in premature newborn brains'
consulta_sin_coincidencias = 'zzzxqvplmok'
assert consulta_nueva not in queries_dict.values()

with socket.socket() as puerto_local:
    puerto_local.bind(('127.0.0.1', 0))
    puerto = puerto_local.getsockname()[1]
url = f'http://127.0.0.1:{puerto}'
cliente = build_opener(ProxyHandler({}))


def solicitud(ruta, datos=None):
    cuerpo = json.dumps(datos).encode('utf-8') if datos is not None else None
    peticion = Request(url + ruta, data=cuerpo, headers={'Content-Type': 'application/json'})
    try:
        with cliente.open(peticion, timeout=5) as respuesta:
            return respuesta.status, json.load(respuesta)
    except HTTPError as error:
        with error:
            return error.code, json.load(error)


def buscar_http(query, model, top_k):
    status, respuesta = solicitud('/search', {'query': query, 'model': model, 'top_k': top_k})
    assert status == 200 and set(respuesta) == {'model', 'results'}, respuesta
    assert respuesta['model'] == model
    resultados = respuesta['results']
    assert len(resultados) <= top_k
    ids = [r['document_id'] for r in resultados]
    assert len(set(ids)) == len(ids)
    puntajes = [r['score'] for r in resultados]
    assert puntajes == sorted(puntajes, reverse=True)
    for rank, resultado in enumerate(resultados, start=1):
        assert set(resultado) == {'rank', 'document_id', 'score', 'title', 'snippet'}
        did = resultado['document_id']
        assert resultado['rank'] == rank and did in corpus_ids
        assert math.isfinite(resultado['score']) and resultado['score'] > 0
        assert resultado['title'] == doc_titles[did]
        assert len(resultado['snippet']) <= 240
        assert doc_texts[did].startswith(resultado['snippet'].removesuffix('…'))
    return resultados


entorno = dict(os.environ, HF_HUB_OFFLINE='1', HF_DATASETS_OFFLINE='1', PYTHONDONTWRITEBYTECODE='1')
with tempfile.TemporaryFile() as registro:
    servidor = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1', '--port', str(puerto)],
        cwd=despliegue,
        env=entorno,
        stdout=registro,
        stderr=subprocess.STDOUT,
    )
    try:
        limite = time.monotonic() + 30
        while True:
            if servidor.poll() is not None or time.monotonic() >= limite:
                raise RuntimeError('Uvicorn no completó el arranque')
            try:
                status, respuesta_health = solicitud('/health')
                break
            except URLError:
                time.sleep(0.2)
        assert status == 200 and respuesta_health == {'status': 'ok', 'index_loaded': True, 'documents': len(doc_ids)}
        print(f'GET /health → {status} {respuesta_health}')

        for modelo, esperado in rankings_originales.items():
            resultados = buscar_http(consulta_original, modelo, 3)
            assert [r['document_id'] for r in resultados] == [did for did, _ in esperado]
            for resultado, (_, puntaje) in zip(resultados, esperado):
                assert math.isclose(resultado['score'], puntaje, rel_tol=0, abs_tol=1e-12)
            ranking = [(r['document_id'], r['score']) for r in resultados]
            print(f'{modelo} | Top 3 original reproducido: {ranking}')

            nuevos = buscar_http(consulta_nueva, modelo, 2)
            assert len(nuevos) == 2
            print(f"{modelo} | Consulta nueva: {[(r['document_id'], r['score']) for r in nuevos]}")

            assert buscar_http(consulta_sin_coincidencias, modelo, 10) == []
            print(f'{modelo} | Consulta sin coincidencias: []')

        assert buscar_http(consulta_original, 'bm25', 0) == []
        assert solicitud('/search', {'query': consulta_original, 'model': 'otro', 'top_k': 3})[0] == 422
        assert solicitud('/search', {'query': consulta_original, 'model': 'bm25', 'top_k': -1})[0] == 422
        print('top_k=0: []; modelo inválido y top_k negativo: HTTP 422')
    except Exception:
        registro.seek(0)
        print(registro.read().decode('utf-8', errors='replace'))
        raise
    finally:
        if servidor.poll() is None:
            servidor.terminate()
            servidor.wait(timeout=10)

GET /health → 200 {'status': 'ok', 'index_loaded': True, 'documents': 5183}
tfidf | Top 3 original reproducido: [('10906636', 0.10632252789889068), ('10608397', 0.10197873840324424), ('31543713', 0.09677198724941677)]
tfidf | Consulta nueva: [('4983', 0.25714204424976395), ('10518721', 0.20028139304547532)]
tfidf | Consulta sin coincidencias: []
bm25 | Top 3 original reproducido: [('21257564', 9.749843781504081), ('18953920', 9.534339229867992), ('40212412', 9.224695024001619)]
bm25 | Consulta nueva: [('4983', 24.94392006707147), ('39390206', 17.757023829590626)]
bm25 | Consulta sin coincidencias: []
top_k=0: []; modelo inválido y top_k negativo: HTTP 422


### Arranque local y ejemplo para Postman

Desde la carpeta del notebook, ejecuta estos comandos en la terminal del entorno Python. Puedes cerrar Jupyter antes de arrancar Uvicorn.

```bash
cd despliegue
python -m pip install -r requirements.txt
python -m uvicorn main:app
```

En Postman, envía `GET http://127.0.0.1:8000/health`. Para buscar, selecciona `POST http://127.0.0.1:8000/search`, abre **Body → raw → JSON** y pega este cuerpo. Cambia `bm25` por `tfidf` para probar el otro modelo.

```json
{
  "query": "0-dimensional biomaterials show inductive properties.",
  "model": "bm25",
  "top_k": 3
}
```